In [112]:
import pandas as pd
import numpy as np

In [113]:
def inspect(df, view='all'):
    """
    Muestra tablas para inspeccionar el DataFrame.
    
    Parámetros
    ----------
    df : pd.DataFrame
    view : str, default 'all'
        - 'all' -> tabla resumen de columnas + tabla de duplicados
        - nombre de columna -> value_counts completo de esa columna
    """
    from IPython.display import display
    if view == 'all':
        desc = df.describe(include="all").T
        desc["n_unique"] = df.nunique()
        desc["null_count"] = df.isnull().sum()
        desc["skewness"] = df.skew(numeric_only=True)
        desc["kurtosis"] = df.kurtosis(numeric_only=True)
        desc["type"] = df.dtypes
        
        try:
            desc = desc.drop(columns=["unique", "top", "freq"])
        except:
            pass
        
        orden = ["type", "count", "null_count", "n_unique",
                 "mean", "std", "min", "25%", "50%", "75%", "max",
                 "skewness", "kurtosis"]
        desc = desc[[c for c in orden if c in desc.columns]]
        
        print("=" * 60 + " RESUMEN DE COLUMNAS " + "=" * 60)
        display(desc)
        
        dups = df[df.duplicated(keep=False)].copy()
        if not dups.empty:
            dups = dups.sort_values(by=list(df.columns))
            print("\n" + "=" * 60 + " FILAS DUPLICADAS (muestra hasta 50) " + "=" * 60)
            display(dups.head(50))
            if len(dups) > 50:
                print(f"... y {len(dups)-50} filas duplicadas más.")
        else:
            print("\n--- No hay filas duplicadas ---")
        
        return None
    
    else:
        if view not in df.columns:
            print(f"La columna '{view}' no existe en el DataFrame.")
            return None
        
        serie = df[view]
        print(f"\nColumna: {view}")
        print(f"Tipo: {serie.dtype}")
        print(f"Nulos: {serie.isnull().sum()} ({serie.isnull().mean():.2%})")
        print(f"Valores únicos: {serie.nunique()}\n")
        
        vc = serie.value_counts(dropna=False)
        vc_df = vc.reset_index()
        vc_df.columns = [view, 'count']
        vc_df['percentage'] = (100 * vc_df['count'] / len(df)).round(2)
        
        if len(vc_df) > 200:
            print(f"(La columna tiene {len(vc_df)} valores únicos, puede tardar en mostrarse)")
        
        with pd.option_context('display.max_rows', None):
            display(vc_df)
        
        return None

In [139]:
campanas_df = pd.read_csv(r"..\01_datos\_raw\campanas.csv")
campanas_df

,fecha,campana_id,canal,tipo_campana,objetivo,audiencia,impresiones,clics,ctr,cpc_mxn,gasto_mxn,conversiones,valor_conv_mxn,roas,dia_semana
0,2024-11-10,FB-AWA-002,Facebook,Awareness,Alcance,Lookalike 1% compradores anteriores,6267,85,0.01356,5.45,467.15,0,0.00,0.00,Sunday
1,2024-06-18,GG-SRC-003,Google,Search,Clics,Intención: como perder peso en casa,412,36,0.08723,19.36,707.76,2,1256.68,1.78,Tuesday
2,2024-05-26,GG-DSP-001,Google,Display,Impresiones,Afinidad: bienestar y fitness,14286,179,0.01253,3.27,586.70,2,1465.95,2.50,Sunday
3,2024-04-04,FB-CVR-001,Facebook,Conversion,Compras,Retargeting carrito abandonado,2601,169,0.06496,9.85,1666.62,9,7784.51,4.67,Thursday
4,2024-02-12,FB-CON-002,Facebook,Consideracion,Trafico,"Intereses: yoga, meditacion, fitness",3376,132,0.03909,7.17,949.99,4,2596.36,2.73,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4286,2024-10-11,FB-AWA-002,Facebook,Awareness,Alcance,Lookalike 1% compradores anteriores,10929,137,0.01253,4.23,580.60,0,0.00,0.00,Friday
4287,2024-04-07,GG-DSP-001,Google,Display,Impresiones,Afinidad: bienestar y fitness,14161,160,0.01130,2.95,474.60,1,909.91,1.92,Sunday
4288,2024-08-03,GG-SRC-003,Google,Search,Clics,Intención: como perder peso en casa,312,19,0.06084,19.21,379.52,0,0.00,0.00,Saturday
4289,2024-11-02,GG-DSP-001,Google,Display,Impresiones,Afinidad: bienestar y fitness,19312,113,0.00585,3.57,405.64,0,0.00,0.00,Saturday


In [140]:
verificacion_campanas_campanas_df = campanas_df.groupby('campana_id')[['objetivo', 'audiencia']].nunique()
verificacion_campanas_campanas_df

,objetivo,audiencia
campana_id,,
FB-AWA-001,1,1
FB-AWA-002,1,1
FB-CON-001,1,1
FB-CON-002,1,1
FB-CVR-001,1,1
FB-CVR-002,1,1
GG-DSP-001,1,1
GG-DSP-002,1,1
GG-SRC-001,1,1


In [141]:
campanas_df['fecha'] = pd.to_datetime(campanas_df['fecha'], format="mixed")
campanas_df['canal'] = campanas_df['canal'].str.lower().replace({
    'facebook': 'Meta', 'fb': 'Meta', 'meta': 'Meta',
    'google': 'Google', 'google ads': 'Google', 'g ads': 'Google'
})
campanas_df['ctr'] = campanas_df['ctr'].fillna(round(campanas_df['clics'] / campanas_df['impresiones'], 5))
campanas_df['objetivo'] = campanas_df.groupby('campana_id')['objetivo'].transform(lambda x: x.ffill().bfill())
campanas_df['audiencia'] = campanas_df.groupby('campana_id')['audiencia'].transform(lambda x: x.ffill().bfill())
campanas_df = campanas_df.drop_duplicates()

In [142]:
inspect(campanas_df, "all")

============================================================ RESUMEN DE COLUMNAS ============================================================


,type,count,null_count,n_unique,mean,std,min,25%,50%,75%,max,skewness,kurtosis
fecha,datetime64[ns],4258,0,366,2024-07-01 18:31:37.228745984,NaN,2024-01-01 00:00:00,2024-04-04 00:00:00,2024-07-02 00:00:00,2024-09-29 00:00:00,2024-12-31 00:00:00,NaN,NaN
campana_id,object,4258,0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal,object,4258,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_campana,object,4258,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
objetivo,object,4258,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
audiencia,object,4258,0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
impresiones,int64,4258.0,0,3297,6112.536872,6261.604118,218.0,903.25,3681.0,10151.5,35848.0,1.185526,0.634404
clics,int64,4258.0,0,306,111.970409,63.782437,10.0,60.0,106.0,152.0,414.0,0.746975,0.405734
ctr,float64,4258.0,0,3280,0.041103,0.03208,0.00378,0.015232,0.03165,0.058167,0.18754,1.175548,0.993853
cpc_mxn,float64,4258.0,0,1578,9.632304,6.0733,1.8,4.42,7.015,14.99,25.47,0.612118,-1.046609



--- No hay filas duplicadas ---


In [143]:
campanas_df.to_csv(r"..\01_datos\processed\campanas.csv", index=False)

In [119]:
conversiones_df = pd.read_csv(r"..\01_datos\_raw\conversiones.csv")
conversiones_df

,usuario_id,campana_id,canal,tipo_campana,fecha,timestamp,etapa,dispositivo
0,U0001000,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 14:01:42,sesion,NaN
1,U0001000,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 14:09:42,vista_producto,movil
2,U0001001,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 23:25:43,sesion,movil
3,U0001002,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 07:07:56,sesion,movil
4,U0001003,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 15:02:06,sesion,movil
...,...,...,...,...,...,...,...,...
831885,U0472840,FB-CON-002,Facebook,Consideracion,2024-08-25,2024-08-25 16:50:31,sesion,desktop
831886,U0090398,FB-CON-001,Facebook,Consideracion,2024-06-08,2024-06-08 20:43:52,sesion,movil
831887,U0080253,GG-SRC-002,Google,Search,2024-11-22,2024-11-22 13:21:49,sesion,movil
831888,U0094951,FB-AWA-001,Facebook,Awareness,2024-02-27,2024-02-27 08:53:46,compra,movil


In [120]:
verificacion_campanas_conversiones_df = conversiones_df.groupby('campana_id')[['tipo_campana']].nunique()
verificacion_campanas_conversiones_df

,tipo_campana
campana_id,
FB-AWA-001,1
FB-AWA-002,1
FB-CON-001,1
FB-CON-002,1
FB-CVR-001,1
FB-CVR-002,1
GG-DSP-001,1
GG-DSP-002,1
GG-SRC-001,1


In [121]:
verificacion_dispositivo_conversiones_df = conversiones_df.groupby('usuario_id')[['dispositivo']].nunique()
verificacion_dispositivo_conversiones_df

,dispositivo
usuario_id,
U0001000,1
U0001001,1
U0001002,1
U0001003,1
U0001004,1
...,...
U0473619,1
U0473620,1
U0473621,1


In [122]:
conversiones_df['canal'] = conversiones_df['canal'].str.lower().replace({
    'facebook': 'Meta', 'fb': 'Meta', 'meta': 'Meta',
    'google': 'Google', 'google ads': 'Google', 'g ads': 'Google'
})
conversiones_df['fecha'] = pd.to_datetime(conversiones_df['fecha'], format="mixed")
conversiones_df['timestamp'] = pd.to_datetime(conversiones_df['timestamp'])
conversiones_df['etapa'] = conversiones_df['etapa'].str.lower().replace({
    'sesion': 'Sesion', 'sesión': 'Sesion', 'vista_producto': 'VistaProducto',
    'vista producto': 'VistaProducto', 'lead': 'Lead', 'checkout': 'Checkout', 
    'compra': 'Purchase', 'vistaprod': 'VistaProducto', 'purchase': 'Purchase',
    'email_capturado': 'EmailCapturado', 'pago_iniciado': 'PagoIniciado'
})
conversiones_df['tipo_campana'] = conversiones_df.groupby('campana_id')['tipo_campana'].transform(lambda x: x.ffill().bfill())
conversiones_df['dispositivo'] = conversiones_df.groupby('usuario_id')['dispositivo'].transform(lambda x: x.ffill().bfill())
conversiones_df['dispositivo'] = conversiones_df['dispositivo'].fillna('Desconocido')
conversiones_df = conversiones_df.drop_duplicates()

C:\Users\markg\AppData\Local\Temp\ipykernel_19852\2049913093.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  conversiones_df['dispositivo'] = conversiones_df.groupby('usuario_id')['dispositivo'].transform(lambda x: x.ffill().bfill())


In [123]:
inspect(conversiones_df, "all")

============================================================ RESUMEN DE COLUMNAS ============================================================


,type,count,null_count,n_unique,mean,min,25%,50%,75%,max,skewness,kurtosis
usuario_id,object,825553,0,472624,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campana_id,object,825553,0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal,object,825553,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_campana,object,825553,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fecha,datetime64[ns],825553,0,366,2024-06-11 00:52:38.551903232,2024-01-01 00:00:00,2024-03-18 00:00:00,2024-05-30 00:00:00,2024-09-10 00:00:00,2024-12-31 00:00:00,NaN,NaN
timestamp,datetime64[ns],825553,0,785233,2024-06-11 14:24:36.787007488,2024-01-01 07:00:20,2024-03-18 13:38:49,2024-05-30 14:42:55,2024-09-10 11:02:12,2025-01-01 00:12:47,NaN,NaN
etapa,object,825553,0,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dispositivo,object,825553,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- No hay filas duplicadas ---


In [124]:
conversiones_df.to_csv(r"..\01_datos\processed\conversiones.csv", index=False)

In [129]:
transacciones_df = pd.read_csv(r"..\01_datos\_raw\transacciones.csv")
transacciones_df

,transaccion_id,usuario_id,campana_id,canal_origen,tipo_campana,fecha,timestamp,producto,categoria,precio_lista_mxn,descuento_mxn,precio_final_mxn,costo_mxn,margen_mxn,metodo_pago,dispositivo
0,TX90001,U0001022,FB-AWA-002,Facebook,NaN,2024-11-10,2024-11-10 18:47:00,Curso Yoga Completo,curso,1499,0.0,1499.0,280,1219.0,tarjeta_debito,movil
1,TX90002,U0001056,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 10:06:00,curso yoga,curso,1499,149.9,1349.1,280,1069.1,tarjeta_debito,movil
2,TX90003,U0001079,FB-AWA-002,Facebook,Awareness,2024-11-10,2024-11-10 20:18:00,MEMBRESIA MENSUAL,membresia,349,0.0,349.0,60,289.0,tarjeta_debito,movil
3,TX90004,U0001106,GG-SRC-003,Google,Search,2024-06-18,2024-06-18 16:57:00,Curso Yoga Completo,curso,1499,0.0,1499.0,280,1219.0,tarjeta_credito,movil
4,TX90005,U0001108,GG-SRC-003,Google,Search,2024-06-18,2024-06-18 09:33:00,Pack Inicio Bienestar,bundle,1299,0.0,1299.0,240,1059.0,tarjeta_debito,desktop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18894,TX99180,U0233045,FB-CVR-001,Facebook,Conversion,2024-12-21,2024-12-21 15:23:00,Curso Fitness en Casa,curso,699,0.0,699.0,130,569.0,tarjeta_credito,movil
18895,TX100318,U0260383,FB-CON-001,Facebook,Consideracion,2024-05-09,2024-05-09 09:31:00,Curso Pilates Basico,curso,599,0.0,599.0,110,489.0,paypal,movil
18896,TX98833,U0225348,GG-SRC-003,Google,Search,2024-05-20,2024-05-20 13:10:00,Membresia Trimestral,membresia,899,0.0,899.0,150,749.0,tarjeta_credito,movil
18897,TX93966,U0099363,FB-AWA-001,Facebook,Awareness,2024-06-28,2024-06-28 12:02:00,Curso Meditacion 21 Dias,curso,799,0.0,799.0,150,649.0,oxxo,desktop


In [130]:
verificacion_campanas_transacciones_df = transacciones_df.groupby('campana_id')[['tipo_campana']].nunique()
verificacion_campanas_transacciones_df

,tipo_campana
campana_id,
FB-AWA-001,1
FB-AWA-002,1
FB-CON-001,1
FB-CON-002,1
FB-CVR-001,1
FB-CVR-002,1
GG-DSP-001,1
GG-DSP-002,1
GG-SRC-001,1


In [131]:
transacciones_df['canal_origen'] = transacciones_df['canal_origen'].str.lower().replace({
    'facebook': 'Meta', 'fb': 'Meta', 'meta': 'Meta',
    'google': 'Google', 'google ads': 'Google', 'g ads': 'Google'
})
transacciones_df['fecha'] = pd.to_datetime(transacciones_df['fecha'], format="mixed")
transacciones_df['timestamp'] = pd.to_datetime(transacciones_df['timestamp'])
transacciones_df['producto'] = transacciones_df['producto'].str.lower().replace({
    'membresia mensual': 'Membresia Mensual', 'membresia mens.': 'Membresia Mensual', 'membresía mensual': 'Membresia Mensual',
    'curso yoga': 'Curso Yoga Completo', 'yoga completo': 'Curso Yoga Completo', 'curso meditacion 21 dias': 'Curso Meditacion 21 Dias',
    'curso yoga completo': 'Curso Yoga Completo', 'curso fitness en casa': 'Curso Fitness En Casa', 'pack inicio bienestar': 'Pack Inicio Bienestar',
    'membresia trimestral': 'Membresia Trimestral', 'curso nutricion funcional': 'Curso Nutricion Funcional', 'curso pilates basico': 'Curso Pilates Basico',
    'taller en vivo mindfulness': 'Taller En Vivo Mindfulness'
})
transacciones_df['categoria'] = transacciones_df['categoria'].str.lower().replace({
    'curso': 'Curso', 'membresia': 'Membresia', 'bundle': 'Bundle',
    'taller': 'Taller', 'membresía': 'Membresia'
})
transacciones_df['precio_lista_mxn'] = transacciones_df['precio_lista_mxn'].astype(float)
transacciones_df['tipo_campana'] = transacciones_df.groupby('campana_id')['tipo_campana'].transform(lambda x: x.ffill().bfill())
transacciones_df['descuento_mxn'] = transacciones_df['descuento_mxn'].fillna(0.0)
transacciones_df['metodo_pago'] = transacciones_df['metodo_pago'].fillna('Desconocido')
transacciones_df = transacciones_df.drop_duplicates()

In [132]:
inspect(transacciones_df, "all")

============================================================ RESUMEN DE COLUMNAS ============================================================


,type,count,null_count,n_unique,mean,std,min,25%,50%,75%,max,skewness,kurtosis
transaccion_id,object,18778,0,18768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
usuario_id,object,18778,0,18768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campana_id,object,18778,0,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal_origen,object,18778,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_campana,object,18778,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fecha,datetime64[ns],18778,0,366,2024-06-09 16:32:27.811268352,NaN,2024-01-01 00:00:00,2024-03-14 00:00:00,2024-05-28 12:00:00,2024-09-09 00:00:00,2024-12-31 00:00:00,NaN,NaN
timestamp,datetime64[ns],18778,0,18193,2024-06-10 07:58:24.925977344,NaN,2024-01-01 08:03:00,2024-03-14 18:25:30,2024-05-29 03:28:30,2024-09-09 12:09:30,2024-12-31 22:50:00,NaN,NaN
producto,object,18778,0,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria,object,18778,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
precio_lista_mxn,float64,18778.0,0,9,820.815422,391.854053,299.0,349.0,799.0,999.0,1499.0,0.372551,-0.951396



--- No hay filas duplicadas ---


In [133]:
transacciones_df.to_csv(r"..\01_datos\processed\transacciones.csv", index=False)